In [1]:
#!/usr/bin/env python3
"""
Comprehensive analysis of the end_user_price raster against demographic
and economic indicators for the Nigerian LPG study.
"""
import os
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio.warp import Resampling, reproject
from scipy.stats import binned_statistic
from matplotlib.ticker import FuncFormatter

# ----------------------------------------------------------------------
# 0. Setup
# ----------------------------------------------------------------------
DATA_DIR = Path("dataset_250526_1138")
PLOT_DIR = DATA_DIR / "analysis_plots"
PLOT_DIR.mkdir(exist_ok=True)

END_USER_PRICE = DATA_DIR / "end_user_price.tif"
POPULATION     = DATA_DIR / "Population.tif"
INCOME         = DATA_DIR / "income_nigeria.tif"
URBAN          = DATA_DIR / "Urban.tif"

BAND_CAR_SHARE     = 0
BAND_WALK_SHARE    = 1
BAND_WALK_TIME     = 3
BAND_CAR_TIME      = 6
BAND_COST_WALK     = 10
BAND_COST_CAR      = 11
BAND_LPG_USE_SHARE = 16
BAND_MAJORITY_COST = 17
BAND_MEAN_COST     = 18

NODATA = -9999.0

# ----------------------------------------------------------------------
# 1. Helper functions
# ----------------------------------------------------------------------
def read_and_align(src_path, ref_profile, resampling=Resampling.bilinear):
    with rasterio.open(src_path) as src:
        arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata
    if src_nodata is not None:
        arr = np.where(arr == src_nodata, np.nan, arr)
    dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
    reproject(
        source=arr,
        destination=dst,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src_nodata,
        dst_transform=ref_profile["transform"],
        dst_crs=ref_profile["crs"],
        dst_nodata=np.nan,
        resampling=resampling,
    )
    return dst

def fmt_num(n):
    """Formats large integer counts to a compact string with a tilde prefix."""
    if np.isnan(n):
        return "0"
    if n >= 1_000_000:
        return f"~{n / 1_000_000:.1f}M"
    elif n >= 1_000:
        return f"~{int(round(n / 1_000))}k"
    return str(int(n))

# Axis formatter function for clear, un-cluttered graph ticks
axis_k_formatter = FuncFormatter(lambda x, pos: f"{int(x * 1e-3)}k" if x >= 1e3 else f"{int(x)}")

# ----------------------------------------------------------------------
# 2. Load reference raster and create mask
# ----------------------------------------------------------------------
print("Loading and aligning rasters...")
with rasterio.open(END_USER_PRICE) as src:
    ref_profile = src.profile.copy()
    full_stack = src.read()

full_stack = np.where(full_stack == NODATA, np.nan, full_stack).astype(np.float32)

pop      = read_and_align(POPULATION, ref_profile)
income   = read_and_align(INCOME, ref_profile)
urban    = read_and_align(URBAN, ref_profile)

income_valid = np.isfinite(income) & (income > 0)
veh_poss = np.full_like(income, np.nan, dtype=np.float32)
if income_valid.any():
    vmin, vmax = income[income_valid].min(), income[income_valid].max()
    if vmax > vmin:
        veh_poss[income_valid] = np.clip((income[income_valid] - vmin) / (vmax - vmin), 0, 1)
    else:
        veh_poss[income_valid] = 0.0

# Exact mask from diagnostic
valid = (
    (pop > 0) &
    np.isfinite(full_stack[BAND_MEAN_COST]) &
    np.isfinite(full_stack[BAND_CAR_SHARE]) &
    np.isfinite(full_stack[BAND_WALK_SHARE])
)
valid_idx = np.where(valid.ravel())[0]

print(f"Total Valid pixels extracted: {len(valid_idx):,}")

# ----------------------------------------------------------------------
# 3. Create full DataFrame (NO SAMPLING)
# ----------------------------------------------------------------------
flat = {
    "mean_cost":     full_stack[BAND_MEAN_COST].ravel()[valid_idx],
    "cost_walk":     full_stack[BAND_COST_WALK].ravel()[valid_idx],
    "cost_car":      full_stack[BAND_COST_CAR].ravel()[valid_idx],
    "majority_cost": full_stack[BAND_MAJORITY_COST].ravel()[valid_idx],
    "car_share":     full_stack[BAND_CAR_SHARE].ravel()[valid_idx],
    "pop":           pop.ravel()[valid_idx],
    "income":        income.ravel()[valid_idx],
    "urban":         urban.ravel()[valid_idx],
    "veh_poss":      veh_poss.ravel()[valid_idx],
    "walk_time_min": full_stack[BAND_WALK_TIME].ravel()[valid_idx],
    "car_time_min":  full_stack[BAND_CAR_TIME].ravel()[valid_idx],
}

df = pd.DataFrame(flat).dropna()
df["is_urban"] = df["urban"] >= 20
df["income_pc"] = df["income"] / df["pop"]
df["income_pc"] = df["income_pc"].clip(upper=np.percentile(df["income_pc"], 99))

print(f"Working dataset initialized with {len(df):,} pixels.")

def save_plot(fig, filename):
    fig.tight_layout()
    fig.savefig(PLOT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Saved: {filename}")

sns.set_theme(style="whitegrid", palette="muted")

# --------------------- 01. Distribution of mean end‑user price ------------
fig, ax = plt.subplots()
sns.kdeplot(df["mean_cost"], fill=True, color="blue", ax=ax)
ax.set_title("Distribution of Mean End-User LPG Price")
ax.set_xlabel("Mean Cost (USD/kg)")
ax.set_ylabel("Density")
save_plot(fig, "01_line_mean_cost.png")

# --------------------- 01X. Distribution of All Cost Types (trimmed x-axis) ----------------
cost_cols = ["mean_cost", "cost_walk", "cost_car", "majority_cost"]
all_costs = pd.concat([df[col] for col in cost_cols])
upper_limit = np.percentile(all_costs, 98)

# Count pixels that have at least one cost exceeding the visible axis
beyond = (df["mean_cost"] > upper_limit) | (df["cost_walk"] > upper_limit) | (df["cost_car"] > upper_limit) | (df["majority_cost"] > upper_limit)
n_beyond = beyond.sum()
pct_beyond = 100 * n_beyond / len(df)

fig, ax = plt.subplots()
sns.kdeplot(df["mean_cost"], fill=True, color="blue", label="Mean Cost", ax=ax)
sns.kdeplot(df["cost_walk"], fill=True, color="orange", label="Walker Cost", ax=ax)
sns.kdeplot(df["cost_car"], fill=True, color="green", label="Driver Cost", ax=ax)
sns.kdeplot(df["majority_cost"], fill=True, color="purple", label="Majority Cost", ax=ax)

ax.set_xlim(0, upper_limit)          # cut the x-axis
ax.set_title("Distribution of LPG Costs by Transportation Mode")
ax.set_xlabel("Cost (USD/kg)")
ax.set_ylabel("Density")
ax.legend()

# Annotation with tilde‑formatted count (e.g. ~90k)
textstr = (f"Pixels beyond {upper_limit:.2f} USD/kg: {fmt_num(n_beyond)}\n"
           f"({pct_beyond:.2f}% of total)")
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.95, 0.95, textstr, transform=ax.transAxes,
        fontsize=9, verticalalignment='top',
        horizontalalignment='right', bbox=props)

save_plot(fig, "01X_line_all_costs.png")

# --------------------- 02. Walker vs Driver cost distributions (Count) ----
fig, ax = plt.subplots()
sns.histplot(df["cost_walk"], label="Walker Cost", element="poly", stat="count", fill=False, color="orange", ax=ax, bins=100)
sns.histplot(df["cost_car"],  label="Driver Cost", element="poly", stat="count", fill=False, color="green", ax=ax, bins=100)
ax.set_title("Cost Distributions: Walker vs Driver (Raw Counts)")
ax.set_xlabel("Cost (USD/kg)")
ax.set_ylabel("Pixel Count")
ax.yaxis.set_major_formatter(axis_k_formatter)

textstr = f"Walker Pixels: {fmt_num(len(df))}\nDriver Pixels: {fmt_num(len(df))}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.95, 0.5, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='center', horizontalalignment='right', bbox=props)
ax.legend()
save_plot(fig, "02_count_walk_vs_car.png")

# --------------------- 03. Mean cost vs Population (binned means) --------- 
fig, ax = plt.subplots()
bins = np.logspace(np.log10(df["pop"].min()), np.log10(df["pop"].max()), 40)
bin_centers = np.sqrt(bins[:-1] * bins[1:])
means, _, _ = binned_statistic(df["pop"], df["mean_cost"], statistic="mean", bins=bins)
stds, _, _  = binned_statistic(df["pop"], df["mean_cost"], statistic="std",  bins=bins)
counts, _, _ = binned_statistic(df["pop"], df["mean_cost"], statistic="count", bins=bins)

mask = counts >= 100
ax.errorbar(bin_centers[mask], means[mask], yerr=stds[mask] / np.sqrt(counts[mask]), 
            fmt="o-", capsize=3, markersize=4, color="teal", label="Mean ± SE")
ax.set_xscale("log")
ax.set_xlabel("Population per pixel")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Population Density (binned)")
ax.legend()
save_plot(fig, "03_mean_cost_vs_pop_binned.png")

# --------------------- 04. Mean cost vs Income per capita (hexbin) --------
fig, ax = plt.subplots()
hb = ax.hexbin(df["income_pc"], df["mean_cost"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.set_xlabel("Income per Capita (USD/month)")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Income per Capita")
save_plot(fig, "04_hexbin_cost_vs_income.png")

# --------------------- 05. Mean cost vs Income + Affordability Threshold ---
fig, ax = plt.subplots()
hb = ax.hexbin(df["income_pc"], df["mean_cost"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")

x_vals = np.linspace(df["income_pc"].min(), df["income_pc"].max(), 100)
y_vals = (0.04 * x_vals) / 58.4
ax.plot(x_vals, y_vals, color='cyan', linewidth=2, linestyle='--', label="Affordability Threshold")

threshold_series = (0.04 * df["income_pc"]) / 58.4
below_thresh = (df["mean_cost"] <= threshold_series).sum()
above_thresh = (df["mean_cost"] > threshold_series).sum()

textstr = f"Sustainable (Below): {fmt_num(below_thresh)}\nToo Expensive (Above): {fmt_num(above_thresh)}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right', bbox=props)

ax.set_xlabel("Income per Capita (USD/year)")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Income w/ Affordability Threshold")
ax.legend(loc="lower left")
save_plot(fig, "05_hexbin_cost_vs_income_threshold.png")

# --------------------- 06. Mean cost vs Car Share (hexbin) ----------------
fig, ax = plt.subplots()
hb = ax.hexbin(df["car_share"], df["mean_cost"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.set_xlabel("Car Share (fraction)")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Car Share")
save_plot(fig, "06_hexbin_cost_vs_carshare.png")

# --------------------- 07. Compare Urban vs Rural: Overlapping Lines ------
fig, ax = plt.subplots()
sns.kdeplot(data=df, x="mean_cost", hue="is_urban", fill=True, common_norm=False, palette={True: "red", False: "blue"}, ax=ax)
ax.set_title("Mean Cost Distribution: Urban vs Rural")
ax.set_xlabel("Mean Cost (USD/kg)")

n_urban = df["is_urban"].sum()
n_rural = len(df) - n_urban
textstr = f"Urban Pixels: {fmt_num(n_urban)}\nRural Pixels: {fmt_num(n_rural)}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.95, 0.5, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='center', horizontalalignment='right', bbox=props)

legend = ax.get_legend()
if legend:
    legend.texts[0].set_text("Rural")
    legend.texts[1].set_text("Urban")
save_plot(fig, "07_compare_cost_urban_rural_kde.png")

# Helper for Maps
def add_elegant_colorbar(im, ax, vmin, vmax, label):
    cbar = plt.colorbar(im, ax=ax, shrink=0.6, label=label)
    mid = (vmin + vmax) / 2
    cbar.set_ticks([vmin, mid, vmax])
    cbar.set_ticklabels([f"{vmin:.2f} (Min)", f"{mid:.2f}", f"{vmax:.2f} (Max)"])

# --------------------- 08. Spatial map of mean cost -----------------------
fig, ax = plt.subplots(figsize=(12, 10))
masked = np.where(valid, full_stack[BAND_MEAN_COST], np.nan)
vmin, vmax = np.nanpercentile(masked, 2), np.nanpercentile(masked, 98)
im = ax.imshow(masked, cmap="RdYlGn_r", vmin=vmin, vmax=vmax)
add_elegant_colorbar(im, ax, vmin, vmax, "USD/kg")
ax.set_title("Spatial Distribution of Mean EndUser Price")
ax.axis("off")
save_plot(fig, "08_map_mean_cost.png")

# --------------------- 09. Spatial map of walker cost ---------------------
fig, ax = plt.subplots(figsize=(12, 10))
masked_w = np.where(valid, full_stack[BAND_COST_WALK], np.nan)
vmin, vmax = np.nanpercentile(masked_w, 2), np.nanpercentile(masked_w, 98)
im = ax.imshow(masked_w, cmap="RdYlGn_r", vmin=vmin, vmax=vmax)
add_elegant_colorbar(im, ax, vmin, vmax, "USD/kg")
ax.set_title("Walker Cost (USD/kg)")
ax.axis("off")
save_plot(fig, "09_map_walk_cost.png")

# --------------------- 10. Spatial map of driver cost ---------------------
fig, ax = plt.subplots(figsize=(12, 10))
masked_c = np.where(valid, full_stack[BAND_COST_CAR], np.nan)
vmin, vmax = np.nanpercentile(masked_c, 2), np.nanpercentile(masked_c, 98)
im = ax.imshow(masked_c, cmap="RdYlGn_r", vmin=vmin, vmax=vmax)
add_elegant_colorbar(im, ax, vmin, vmax, "USD/kg")
ax.set_title("Driver Cost (USD/kg)")
ax.axis("off")
save_plot(fig, "10_map_driver_cost.png")

# --------------------- 11. Spatial map of majority cost -------------------
fig, ax = plt.subplots(figsize=(12, 10))
masked_m = np.where(valid, full_stack[BAND_MAJORITY_COST], np.nan)
vmin, vmax = np.nanpercentile(masked_m, 2), np.nanpercentile(masked_m, 98)
im = ax.imshow(masked_m, cmap="RdYlGn_r", vmin=vmin, vmax=vmax)
add_elegant_colorbar(im, ax, vmin, vmax, "USD/kg")
ax.set_title("Majority Cost (Walker/Driver based on share)")
ax.axis("off")
save_plot(fig, "11_map_majority_cost.png")

# --------------------- 12. Line plot: mean cost within income brackets ----
df['income_pct'] = df['income_pc'].rank(pct=True) * 100
df['income_bin'] = pd.cut(df['income_pct'], bins=np.arange(0, 105, 5), labels=np.arange(2.5, 100, 5))
grouped_inc = df.groupby('income_bin', observed=True)['mean_cost'].mean()

fig, ax = plt.subplots()
ax.plot(grouped_inc.index, grouped_inc.values, marker="o")
ax.set_xlabel("Income Percentile Brackets (5% blocks)")
ax.set_ylabel("Average Mean Cost in Bracket (USD/kg)")
ax.set_title("Mean Cost by Income Brackets")
save_plot(fig, "12_line_cost_vs_income_brackets.png")

# --------------------- 13. Line plot: mean cost vs pop (Log Scale) --------
fig, ax = plt.subplots()
df_g13 = df[df["pop"] >= 1]
n_displayed = len(df_g13)
n_excluded = len(df) - n_displayed

pop_bins = np.logspace(np.log10(df_g13["pop"].min()), np.log10(df_g13["pop"].max()), 20)
pop_bin_centers = np.sqrt(pop_bins[:-1] * pop_bins[1:])
means_pop, _, _ = binned_statistic(df_g13["pop"], df_g13["mean_cost"], statistic="mean", bins=pop_bins)
ax.plot(pop_bin_centers, means_pop, marker="o")
ax.set_xscale("log")
ax.set_xlabel("Population per pixel (Log Scale)")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Logarithmic Population Segments")

# --- moved to bottom‑left and clarified "pixels" ---
textstr = f"Displayed Pixels (Pop ≥ 1): {fmt_num(n_displayed)}\nExcluded Pixels (Pop < 1): {fmt_num(n_excluded)}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.05, 0.05, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', horizontalalignment='left', bbox=props)
save_plot(fig, "13_line_cost_vs_pop_log.png")

# --------------------- 14. Cost within car share brackets -----------------
df['car_pct'] = df['car_share'].rank(pct=True) * 100
df['car_bin'] = pd.cut(df['car_pct'], bins=np.arange(0, 105, 5), labels=np.arange(2.5, 100, 5))
grouped_car = df.groupby('car_bin', observed=True)['mean_cost'].mean()

fig, ax = plt.subplots()
ax.plot(grouped_car.index, grouped_car.values, marker="o", color='blue', label='Mean Cost')
ax.set_xlabel("Car Share Percentile Brackets (5% blocks)")
ax.set_ylabel("Average Mean Cost (USD/kg)", color='blue')
ax.set_title("Mean Cost by Car Share Brackets")
save_plot(fig, "14_line_cost_vs_carshare_brackets.png")

# --------------------- 15. Correlation heatmap ----------------------------
# Ensure log-transformed columns exist (added by diagnostics cell)
if "log_pop" not in df.columns:
    df["log_pop"] = np.log10(df["pop"])
if "log_income_pc" not in df.columns:
    df["log_income_pc"] = np.log10(df["income_pc"].clip(lower=1))

# Select a compact, non-redundant set of variables
heatmap_vars = [
    "mean_cost", "cost_walk", "cost_car",
    "walk_time_min", "car_time_min", "car_share",
    "log_pop", "log_income_pc", "is_urban"
]

# Convert boolean is_urban to int for correlation
df_hm = df[heatmap_vars].copy()
if df_hm["is_urban"].dtype == bool:
    df_hm["is_urban"] = df_hm["is_urban"].astype(int)

corr_matrix = df_hm.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",        # red = positive, blue = negative
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    ax=ax
)
ax.set_title("Correlation Matrix of Key LPG Cost Determinants", fontsize=14, pad=20)
ax.tick_params(labelsize=9)
fig.tight_layout()
save_plot(fig, "15_correlation_heatmap.png")

# --------------------- 16. 2D histogram: walk vs driver (re‑created) ------
fig, ax = plt.subplots()

# Use percentiles to zoom in on the bulk of the data
x1, x99 = np.percentile(df["cost_walk"], 1), np.percentile(df["cost_walk"], 99)
y1, y99 = np.percentile(df["cost_car"], 1), np.percentile(df["cost_car"], 99)
lim_min = min(x1, y1)
lim_max = max(x99, y99)

hb = ax.hexbin(df["cost_walk"], df["cost_car"], gridsize=50, cmap="turbo",
               mincnt=1, norm=matplotlib.colors.LogNorm(),
               extent=[lim_min, lim_max, lim_min, lim_max])
plt.colorbar(hb, label="Count")
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.5, label="y = x")
ax.set_xlabel("Walker Cost (USD/kg)")
ax.set_ylabel("Driver Cost (USD/kg)")
ax.set_title("Walker vs Driver End-User Cost (central 98% data)")
ax.legend()
save_plot(fig, "16_hexbin_walk_vs_car.png")

# --------------------- 17. Cost difference vs car share -------------------
df["cost_diff"] = df["cost_walk"] - df["cost_car"]
fig, ax = plt.subplots()
hb = ax.hexbin(df["car_share"], df["cost_diff"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.axhline(0, color="grey", linestyle="--")
ax.set_xlabel("Car Share")
ax.set_ylabel("Walker Cost − Driver Cost (USD/kg)")
ax.set_title("Cost Advantage of Driving vs Car Share")

walker_higher = (df["cost_diff"] > 0).sum()
driver_higher = (df["cost_diff"] < 0).sum()
textstr = f"Walker Cost Higher: {fmt_num(walker_higher)}\nDriver Cost Higher: {fmt_num(driver_higher)}"
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.95, 0.05, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', horizontalalignment='right', bbox=props)
save_plot(fig, "17_hexbin_cost_diff_vs_carshare.png")

# --------------------- 18. Mean cost vs Vehicle Possibility ---------------
fig, ax = plt.subplots()
hb = ax.hexbin(df["veh_poss"], df["mean_cost"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.set_xlabel("Vehicle Possibility (normalized)")
ax.set_ylabel("Mean Cost (USD/kg)")
ax.set_title("Mean Cost vs Vehicle Possibility")
save_plot(fig, "18_hexbin_cost_vs_veh_poss.png")

# --------------------- 19. Cost vs travel time (walker) -------------------
fig, ax = plt.subplots()
hb = ax.hexbin(df["walk_time_min"], df["cost_walk"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.set_xlabel("Walking Time to Reseller (min)")
ax.set_ylabel("Walker Cost (USD/kg)")
ax.set_title("Walker Cost vs Walking Time")
save_plot(fig, "19_hexbin_cost_walk_vs_time.png")

# --------------------- 20. Cost vs travel time (driver) -------------------
fig, ax = plt.subplots()
hb = ax.hexbin(df["car_time_min"], df["cost_car"], gridsize=50, cmap="turbo", mincnt=1, norm=matplotlib.colors.LogNorm())
plt.colorbar(hb, label="Count")
ax.set_xlabel("Driving Time to Reseller (min)")
ax.set_ylabel("Driver Cost (USD/kg)")
ax.set_title("Driver Cost vs Driving Time")
save_plot(fig, "20_hexbin_cost_car_vs_time.png")

# --------------------- 21. Urban/Rural breakdown (Raw Counts) -------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (is_urban, title) in enumerate([(True, "Urban"), (False, "Rural")]):
    subset = df[df["is_urban"] == is_urban]
    sns.histplot(subset["cost_walk"], element="poly", stat="count", fill=False, label="Walker", color="orange", ax=axes[i], bins=50)
    sns.histplot(subset["cost_car"],  element="poly", stat="count", fill=False, label="Driver", color="green", ax=axes[i], bins=50)
    axes[i].set_title(f"{title}  Cost Distributions (Raw Counts)")
    axes[i].set_xlabel("Cost (USD/kg)")
    axes[i].yaxis.set_major_formatter(axis_k_formatter)
    
    textstr = f"{title} Pixels: {fmt_num(len(subset))}"
    props = dict(boxstyle='round', facecolor='white', alpha=0.9)
    axes[i].text(0.95, 0.5, textstr, transform=axes[i].transAxes, fontsize=10,
                 verticalalignment='center', horizontalalignment='right', bbox=props)
    if i == 0:
        axes[i].legend()
save_plot(fig, "21_count_cost_urban_rural.png")

# --------------------- 22. Spatial map of cost difference -----------------
diff_raster = np.where(valid, full_stack[BAND_COST_WALK] - full_stack[BAND_COST_CAR], np.nan)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(diff_raster, cmap="coolwarm", vmin=-0.5, vmax=0.5)
plt.colorbar(im, shrink=0.6, label="USD/kg")
ax.set_title("Cost Difference: Walker − Driver")
ax.axis("off")
save_plot(fig, "22_map_cost_difference.png")

# --------------------- 23. Scatter with marginal dist (Density Contours) --
jp_df = df.sample(min(10000, len(df)), random_state=42)
g = sns.jointplot(data=jp_df, x="car_share", y="mean_cost", kind="kde", cmap="turbo", fill=True)
g.set_axis_labels("Car Share", "Mean Cost (USD/kg)")
g.fig.suptitle("Mean Cost vs Car Share (Density Contours)", y=1.02)
save_plot(g.fig, "23_jointplot_cost_carshare.png")

print("\nAll 24 valid plots successfully generated.")

Loading and aligning rasters...
Total Valid pixels extracted: 563,851
Working dataset initialized with 456,759 pixels.
   Saved: 01_line_mean_cost.png
   Saved: 01X_line_all_costs.png
   Saved: 02_count_walk_vs_car.png
   Saved: 03_mean_cost_vs_pop_binned.png
   Saved: 04_hexbin_cost_vs_income.png
   Saved: 05_hexbin_cost_vs_income_threshold.png
   Saved: 06_hexbin_cost_vs_carshare.png
   Saved: 07_compare_cost_urban_rural_kde.png
   Saved: 08_map_mean_cost.png
   Saved: 09_map_walk_cost.png
   Saved: 10_map_driver_cost.png
   Saved: 11_map_majority_cost.png
   Saved: 12_line_cost_vs_income_brackets.png
   Saved: 13_line_cost_vs_pop_log.png
   Saved: 14_line_cost_vs_carshare_brackets.png
   Saved: 15_correlation_heatmap.png
   Saved: 16_hexbin_walk_vs_car.png
   Saved: 17_hexbin_cost_diff_vs_carshare.png
   Saved: 18_hexbin_cost_vs_veh_poss.png
   Saved: 19_hexbin_cost_walk_vs_time.png
   Saved: 20_hexbin_cost_car_vs_time.png
   Saved: 21_count_cost_urban_rural.png
   Saved: 22_map_cos